In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
time: 3.06 s (started: 2025-07-14 09:12:46 +00:00)


In [ ]:

%%capture
!pip install ipython-autotime
!pip install sentence-transformers faiss-cpu pymupdf tqdm transformers torch accelerate bitsandbytes
!pip install pytesseract
%load_ext autotime

time: 38.2 s (started: 2025-07-14 09:12:54 +00:00)


time: 21.5 s (started: 2025-07-14 09:00:16 +00:00)


time: 21.5 s (started: 2025-07-14 09:00:16 +00:00)


In [ ]:
from google.colab import userdata
wandb_token = userdata.get('wandb')
huggingface_token = userdata.get('huggingface')


In [ ]:
import wandb
wandb.login(key = wandb_token)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: christeenhallak33 (christeenhallak33-coretech-mena) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!huggingface-cli login --token $huggingface_token

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `huggingfaceToken` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `huggingfaceToken`


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from huggingface_hub import hf_hub_download
from abc import ABC, abstractmethod
# from llama_cpp import Llama
import fitz  # PyMuPDF
from tqdm import tqdm
from PIL import Image
import pytesseract
import numpy as np
import torch
import faiss
import json
import re
import os

time: 1.04 s (started: 2025-07-14 08:59:36 +00:00)


In [ ]:
saved_dir = '/content/gdrive/MyDrive/Artificial Intelligence/LLM/Fine Tunning/'
model = 'google/gemma-3-12b-it'

In [ ]:
import json_repair

In [ ]:
def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

In [ ]:
parse_json("'''json{questions:[]}json'''")

{'questions': []}

In [ ]:
# base_url = "https://lms-be.coretech-mena.com/documents"
# resource_file_url = "worldwar_4a62c96b-aa31-4f56-a8e1-98d3ecb19dc8.pdf"
# resource_file = f'{base_url}/{resource_file_url}'
# import requests
# import os
# import tempfile
# import PyPDF2

# def extract_text_from_pdf(pdf_path_or_url: str) -> str:
#     """Extract full text from a PDF file, supports both local and remote (temp) PDF."""
#     temp_file_path = None

#     try:
#         # Check if it's a URL
#         if pdf_path_or_url.startswith("http://") or pdf_path_or_url.startswith("https://"):
#             response = requests.get(pdf_path_or_url)
#             response.raise_for_status()

#             # Save to a temporary file
#             with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
#                 tmp.write(response.content)
#                 temp_file_path = tmp.name
#             file_path = temp_file_path
#         else:
#             file_path = pdf_path_or_url

#         # Read the PDF using PyPDF2
#         with open(file_path, 'rb') as file:
#             reader = PyPDF2.PdfReader(file)
#             pages = reader.pages
#             pdf = " ".join(page.extract_text() or "" for page in pages).strip()
#             return pdf

#     except Exception as e:
#         return f"Error reading PDF: {e}"

#     finally:
#         # Clean up the temp file if one was created
#         if temp_file_path and os.path.exists(temp_file_path):
#             os.remove(temp_file_path)



# pdf_content = extract_text_from_pdf(resource_file)

In [ ]:
# Step 1: Upload file from your local device
from google.colab import files
uploaded = files.upload()  # Opens file picker

# Step 2: Extract the file path (assumes single file uploaded)
import os

pdf_path = next(iter(uploaded))  # Get the uploaded filename

# Step 3: Use your existing function (adjusted to skip URL handling)
import PyPDF2

def extract_text_from_pdf_local(pdf_path: str) -> str:
    """Extract text from a local PDF file."""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            pages = reader.pages
            pdf_text = " ".join(page.extract_text() or "" for page in pages).strip()
            return pdf_text
    except Exception as e:
        return f"Error reading PDF: {e}"

# Step 4: Extract text from the uploaded PDF
pdf_content_local = extract_text_from_pdf_local(pdf_path)




Saving short_doc.pdf to short_doc.pdf


In [ ]:
cleaned_text = ' '.join(pdf_content_local.split())
print(cleaned_text)

ﺷﮭد ﺗﺎرﯾﺦ ﻣﺻر ﻓﻲ اﻟﻘرن اﻟﻌﺷرﯾن ﺗﺣوﻻت ﺳﯾﺎﺳﯾﺔ واﺟﺗﻣﺎﻋﯾﺔ ﻛﺑﯾرة، ﺑداﯾﺔ ﻣن اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ وﺻوﻻً إﻟﻰ ﻗﯾﺎم اﻟﺛورة ﻋﺎم 1952 وﺗﺣول ﻣﺻر إﻟﻰ ﺟﻣﮭورﯾﺔ. ﺗﻣﯾز ھذا اﻟﻘرن ﺑظﮭور اﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ اﻟﻣﺻرﯾﺔ، اﻟﻣطﺎﻟﺑﺎت ﺑﺎﻻﺳﺗﻘﻼل، وﺗﺄﺳﯾس أﺣزاب ﺳﯾﺎﺳﯾﺔ، ﺑﺎﻹﺿﺎﻓﺔ إﻟﻰ ﺗطورات اﺟﺗﻣﺎﻋﯾﺔ وﺛﻘﺎﻓﯾﺔ ﻣﮭﻣﺔ. اﻟﻔﺗرة اﻷوﻟﻰ ) 1900 - 1922 :( اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ واﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ • ﺑدأت ﻣﺻر اﻟﻘرن اﻟﻌﺷرﯾن ﺗﺣت اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ، اﻟذي ﺗﺳﺑب ﻓﻲ ﺿﻐوط اﻗﺗﺻﺎدﯾﺔ وﺳﯾﺎﺳﯾﺔ ﻛﺑﯾرة. • ﺗﺻﺎﻋدت اﻟﻣﻘﺎوﻣﺔ اﻟﺷﻌﺑﯾﺔ واﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ ﺿد اﻻﺣﺗﻼل، ﺑزﻋﺎﻣﺔ ﻣﺻطﻔﻰ ﻛﺎﻣل وﻣﺣﻣد ﻓرﯾد. • ظﮭرت ﺛورة 1919 ﻟﻠﻣطﺎﻟﺑﺔ ﺑﺎﻻﺳﺗﻘﻼل، وﻛﺎن ﻟﻠزﻋﯾم ﺳﻌد زﻏﻠول دور ﺑﺎرز ﻓﯾﮭﺎ. • ﻓﻲ ﻋﺎم 1922 ، ﺗم إﻋﻼن اﺳﺗﻘﻼل ﻣﺻر، ﻟﻛن اﻟﻧﻔوذ اﻟﺑرﯾطﺎﻧﻲ ظل ﻛﺑﯾرًا. اﻟﻔﺗرة اﻟﺛﺎﻧﯾﺔ ) 1922 - 1952 :( اﻟﺣﻛم اﻟﻣﻠﻛﻲ واﻟﻧﮭﺿﺔ اﻟﺛﻘﺎﻓﯾﺔ


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict, Literal


# --- Property choices ---
DifficultyLevel = Literal["Very Easy", "Easy", "Average", "Difficult"]
BloomTaxonomy = Literal["Remember", "Apply", "Evaluate"]
LanguageType = Literal["Arabic", "English"]
ResponseTime = Literal["Short", "Medium", "Long"]



# --- Properties for a single question ---
class QuestionProperties(BaseModel):
    difficulty: DifficultyLevel = Field(..., alias="Difficulty")
    bloom_taxonomy: BloomTaxonomy = Field(..., alias="Bloom Taxonomy")
    language: LanguageType = Field(..., alias="Language")
    response_time: ResponseTime = Field(..., alias="Response Time")


class Question(BaseModel):
    question_text: str = Field(..., description="Question text.")
    question_answers: List[str] = Field(..., min_items=2, description="List of answer options.")
    correct_answer: str = Field(..., description="The correct answer.")
    properties: QuestionProperties = Field(..., description="Per-question attributes.")


# --- Full question set schema ---
class QuestionSet(BaseModel):
    questions: List[Question] = Field(..., description="List of MCQs.")


In [ ]:
def generate_prompt(number_of_questions, question_type, number_of_choices, properties, language):
    QUESTION_TYPE_MAP = {
        0: "multiple-choice questions (one correct answer)",
        1: "multiple-choice questions (multiple correct answers)",
        2: "true/false questions",
        3: "essay questions"
    }

    property_distributions = {}
    question_properties_lines = []
    total_questions_assigned = 0
    processed_items = []

    # First pass: calculate question counts from percentages and collect fixed values
    for prop_name, prop_value in properties.items():
        if isinstance(prop_value, list) and all(isinstance(item, dict) for item in prop_value):
            dist_parts = []
            enum_parts = []
            for item in prop_value:
                for k, v in item.items():
                    if isinstance(v, (int, float)):
                        count = int((v * number_of_questions) / 100)
                        processed_items.append((prop_name, k, v, count))
                        total_questions_assigned += count
                    else:
                        # Handle non-percentage items if necessary, although the current logic assumes percentage for lists of dicts
                         pass
            if dist_parts:
                property_distributions[prop_name] = ", ".join(dist_parts)
                question_properties_lines.append(
                    f'       "{prop_name}": {" | ".join([k.strip("") for k in enum_parts])}'
                )
        else:
            # Fixed value
            property_distributions[prop_name] = str(prop_value)
            question_properties_lines.append(f'       "{prop_name}": "{prop_value}"')


    # Adjust remainder to ensure total = number_of_questions for percentage-based distributions
    remainder = number_of_questions - total_questions_assigned
    if processed_items and remainder > 0:
        # Find the item with the largest initial count to add the remainder to
        processed_items.sort(key=lambda x: x[3], reverse=True)
        prop_name, k, v, count = processed_items[0]
        processed_items[0] = (prop_name, k, v, count + remainder)

    # Rebuild distribution strings and property lines based on adjusted counts
    property_distributions = {}
    question_properties_lines = []

    for prop_name, prop_value in properties.items():
         if isinstance(prop_value, list) and all(isinstance(item, dict) for item in prop_value):
            dist_parts = []
            enum_parts = []
            for item_prop_name, k, v, count in processed_items:
                if item_prop_name == prop_name:
                    if count > 0:
                         dist_parts.append(f'{k} ({count})')
                         enum_parts.append(f'"{k}"')
            if dist_parts:
                property_distributions[prop_name] = ", ".join(dist_parts)
                question_properties_lines.append(
                    f'       "{prop_name}": {" | ".join([k.strip("") for k in enum_parts])}'
                )
         else:
            property_distributions[prop_name] = str(prop_value)
            question_properties_lines.append(f'       "{prop_name}": "{prop_value}"')


    # Section for distribution description
    distribution_lines = "\n".join([
        f"   - {prop}: {desc}" for prop, desc in property_distributions.items() if prop != "extra_properties"
    ])

    # Properties block inside each question
    question_properties_block = ",\n".join(question_properties_lines)
    question_type_str = QUESTION_TYPE_MAP.get(question_type, "questions")

    # Final prompt
    prompt = f"""
You are a question generation expert.
from this text {{{{text}}}},
generate exactly {number_of_questions} {question_type_str}.
Follow these instructions precisely:

1. Distribute the {number_of_questions} questions according to these constraints:
{distribution_lines}
follow the distribution proportions strictly. Use reasonable assumptions if the text lacks enough variety.

2. Each question must include:
   - "question_text": (string)
   - "question_answers": (list of {number_of_choices} choices)
   - "correct_answer": (one of the above choices)
   - "properties": an object that includes:
{question_properties_block}

3. Language for all question_text and answers must be {language}.

4. You have to extract JSON details from pdf, containing list of {number_of_questions} question objects.
Do NOT include any explanation, formatting, or text outside the JSON object.

⚠️ Ensure the distribution rules are strictly followed. Extract content accurately and comply with the format.
""".strip()

    return prompt

In [ ]:
user_prompt = generate_prompt(number_of_questions = 2,question_type=0,number_of_choices=2,language='Arabic',properties={
    "Difficulty":[{"Easy":30,"Average":40,"Difficult":30}],
    "Bloom Taxonomy":[{"Apply":20,"Remember":30,"Evaluate":50}],
    "Response Time":[{"Short":40,"Medium":20,"Long":40}]
  })

In [ ]:
user_prompt

'You are a question generation expert.\nfrom this text {{text}},\ngenerate exactly 2 multiple-choice questions (one correct answer).\nFollow these instructions precisely:\n\n1. Distribute the 2 questions according to these constraints:\n   - Bloom Taxonomy: Evaluate (2)\nfollow the distribution proportions strictly. Use reasonable assumptions if the text lacks enough variety.\n\n2. Each question must include:\n   - "question_text": (string)\n   - "question_answers": (list of 2 choices)\n   - "correct_answer": (one of the above choices)\n   - "properties": an object that includes:\n       "Bloom Taxonomy": "Evaluate"\n\n3. Language for all question_text and answers must be Arabic.\n\n4. You have to extract JSON details from pdf, containing list of 2 question objects.\nDo NOT include any explanation, formatting, or text outside the JSON object.\n\n⚠️ Ensure the distribution rules are strictly followed. Extract content accurately and comply with the format.'

In [ ]:
import torch
dtype = torch.bfloat16
quantization_config = BitsAndBytesConfig(
          load_in_4bit=True,
          bnb_4bit_compute_dtype=torch.float16
      )
model_name = 'google/gemma-7b-it'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=quantization_config
                                             ,torch_dtype=dtype,
                                             device_map="auto",)


PackageNotFoundError: No package metadata was found for bitsandbytes

In [ ]:
user_prompt = generate_prompt(number_of_questions = 10,question_type=0,number_of_choices=4,language='Arabic',properties={
    "Difficulty":[{"Easy":30,"Average":40,"Difficult":30}],
    "Bloom Taxonomy":[{"Apply":20,"Remember":30,"Evaluate":50}],
    "Response Time":[{"Short":40,"Medium":20,"Long":40}]
  })


user_prompt = user_prompt.replace("{{text}}",cleaned_text)
chat = [
    {
        "role":"user",
        "content":f"{user_prompt}"
    }
]


In [ ]:
user_prompt

'You are a question generation expert.\nfrom this text ﺷﮭد ﺗﺎرﯾﺦ ﻣﺻر ﻓﻲ اﻟﻘرن اﻟﻌﺷرﯾن ﺗﺣوﻻت ﺳﯾﺎﺳﯾﺔ واﺟﺗﻣﺎﻋﯾﺔ ﻛﺑﯾرة، ﺑداﯾﺔ ﻣن اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ وﺻوﻻً إﻟﻰ ﻗﯾﺎم اﻟﺛورة ﻋﺎم 1952 وﺗﺣول ﻣﺻر إﻟﻰ ﺟﻣﮭورﯾﺔ. ﺗﻣﯾز ھذا اﻟﻘرن ﺑظﮭور اﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ اﻟﻣﺻرﯾﺔ، اﻟﻣطﺎﻟﺑﺎت ﺑﺎﻻﺳﺗﻘﻼل، وﺗﺄﺳﯾس أﺣزاب ﺳﯾﺎﺳﯾﺔ، ﺑﺎﻹﺿﺎﻓﺔ إﻟﻰ ﺗطورات اﺟﺗﻣﺎﻋﯾﺔ وﺛﻘﺎﻓﯾﺔ ﻣﮭﻣﺔ. اﻟﻔﺗرة اﻷوﻟﻰ ) 1900 - 1922 :( اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ واﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ • ﺑدأت ﻣﺻر اﻟﻘرن اﻟﻌﺷرﯾن ﺗﺣت اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ، اﻟذي ﺗﺳﺑب ﻓﻲ ﺿﻐوط اﻗﺗﺻﺎدﯾﺔ وﺳﯾﺎﺳﯾﺔ ﻛﺑﯾرة. • ﺗﺻﺎﻋدت اﻟﻣﻘﺎوﻣﺔ اﻟﺷﻌﺑﯾﺔ واﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ ﺿد اﻻﺣﺗﻼل، ﺑزﻋﺎﻣﺔ ﻣﺻطﻔﻰ ﻛﺎﻣل وﻣﺣﻣد ﻓرﯾد. • ظﮭرت ﺛورة 1919 ﻟﻠﻣطﺎﻟﺑﺔ ﺑﺎﻻﺳﺗﻘﻼل، وﻛﺎن ﻟﻠزﻋﯾم ﺳﻌد زﻏﻠول دور ﺑﺎرز ﻓﯾﮭﺎ. • ﻓﻲ ﻋﺎم 1922 ، ﺗم إﻋﻼن اﺳﺗﻘﻼل ﻣﺻر، ﻟﻛن اﻟﻧﻔوذ اﻟﺑرﯾطﺎﻧﻲ ظل ﻛﺑﯾرًا. اﻟﻔﺗرة اﻟﺛﺎﻧﯾﺔ ) 1922 - 1952 :( اﻟﺣﻛم اﻟﻣﻠﻛﻲ واﻟﻧﮭﺿﺔ اﻟﺛﻘﺎﻓﯾﺔ,\ngenerate exactly 10 multiple-choice questions (one correct answer).\nFollow these instructions precisely:\n\n1. Distribute the 10 questions according to these constraints:\n   - Difficulty: Easy (3), Average (4), Difficult (3)\n   - Blo

In [ ]:
prompt = tokenizer.apply_chat_template(chat,tokenize = False,add_generation_prompt = True)
inputs = tokenizer(prompt,add_special_tokens = False,return_tensors="pt").to(model.device)

In [ ]:
prompt

'<bos><start_of_turn>user\nYou are a question generation expert.\nfrom this text ﺷﮭد ﺗﺎرﯾﺦ ﻣﺻر ﻓﻲ اﻟﻘرن اﻟﻌﺷرﯾن ﺗﺣوﻻت ﺳﯾﺎﺳﯾﺔ واﺟﺗﻣﺎﻋﯾﺔ ﻛﺑﯾرة، ﺑداﯾﺔ ﻣن اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ وﺻوﻻً إﻟﻰ ﻗﯾﺎم اﻟﺛورة ﻋﺎم 1952 وﺗﺣول ﻣﺻر إﻟﻰ ﺟﻣﮭورﯾﺔ. ﺗﻣﯾز ھذا اﻟﻘرن ﺑظﮭور اﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ اﻟﻣﺻرﯾﺔ، اﻟﻣطﺎﻟﺑﺎت ﺑﺎﻻﺳﺗﻘﻼل، وﺗﺄﺳﯾس أﺣزاب ﺳﯾﺎﺳﯾﺔ، ﺑﺎﻹﺿﺎﻓﺔ إﻟﻰ ﺗطورات اﺟﺗﻣﺎﻋﯾﺔ وﺛﻘﺎﻓﯾﺔ ﻣﮭﻣﺔ. اﻟﻔﺗرة اﻷوﻟﻰ ) 1900 - 1922 :( اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ واﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ • ﺑدأت ﻣﺻر اﻟﻘرن اﻟﻌﺷرﯾن ﺗﺣت اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ، اﻟذي ﺗﺳﺑب ﻓﻲ ﺿﻐوط اﻗﺗﺻﺎدﯾﺔ وﺳﯾﺎﺳﯾﺔ ﻛﺑﯾرة. • ﺗﺻﺎﻋدت اﻟﻣﻘﺎوﻣﺔ اﻟﺷﻌﺑﯾﺔ واﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ ﺿد اﻻﺣﺗﻼل، ﺑزﻋﺎﻣﺔ ﻣﺻطﻔﻰ ﻛﺎﻣل وﻣﺣﻣد ﻓرﯾد. • ظﮭرت ﺛورة 1919 ﻟﻠﻣطﺎﻟﺑﺔ ﺑﺎﻻﺳﺗﻘﻼل، وﻛﺎن ﻟﻠزﻋﯾم ﺳﻌد زﻏﻠول دور ﺑﺎرز ﻓﯾﮭﺎ. • ﻓﻲ ﻋﺎم 1922 ، ﺗم إﻋﻼن اﺳﺗﻘﻼل ﻣﺻر، ﻟﻛن اﻟﻧﻔوذ اﻟﺑرﯾطﺎﻧﻲ ظل ﻛﺑﯾرًا. اﻟﻔﺗرة اﻟﺛﺎﻧﯾﺔ ) 1922 - 1952 :( اﻟﺣﻛم اﻟﻣﻠﻛﻲ واﻟﻧﮭﺿﺔ اﻟﺛﻘﺎﻓﯾﺔ,\ngenerate exactly 10 multiple-choice questions (one correct answer).\nFollow these instructions precisely:\n\n1. Distribute the 10 questions according to these constraints:\n   - Difficulty: Easy (3), Average (4

In [ ]:
# inputs

In [ ]:
outputs = model.generate(input_ids=inputs["input_ids"].to(model.device),
                              max_new_tokens=2048)
generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]

KeyboardInterrupt: 

In [ ]:
def format_response(response: str) -> list:
    try:
        # Extract JSON from markdown code blocks
        start = response.find('```json') + len('```json')
        end = response.find('```', start)
        json_content = response[start:end].strip()
        return json.loads(response[start:end])
    except json.JSONDecodeError:
        raise ValueError("Failed to parse model response")

In [ ]:
full_response = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
print(full_response)
format_response(full_response)

In [ ]:
print(full_response)

<bos><start_of_turn>user
You are a question generation expert.
from this ﺷﮭد ﺗﺎرﯾﺦ ﻣﺻر ﻓﻲ اﻟﻘرن اﻟﻌﺷرﯾن ﺗﺣوﻻت ﺳﯾﺎﺳﯾﺔ واﺟﺗﻣﺎﻋﯾﺔ ﻛﺑﯾرة، ﺑداﯾﺔ ﻣن اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ وﺻوﻻً إﻟﻰ ﻗﯾﺎم اﻟﺛورة ﻋﺎم 1952 وﺗﺣول ﻣﺻر إﻟﻰ ﺟﻣﮭورﯾﺔ. ﺗﻣﯾز ھذا اﻟﻘرن ﺑظﮭور اﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ اﻟﻣﺻرﯾﺔ، اﻟﻣطﺎﻟﺑﺎت ﺑﺎﻻﺳﺗﻘﻼل، وﺗﺄﺳﯾس أﺣزاب ﺳﯾﺎﺳﯾﺔ، ﺑﺎﻹﺿﺎﻓﺔ إﻟﻰ ﺗطورات اﺟﺗﻣﺎﻋﯾﺔ وﺛﻘﺎﻓﯾﺔ ﻣﮭﻣﺔ. اﻟﻔﺗرة اﻷوﻟﻰ ) 1900 - 1922 :( اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ واﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ • ﺑدأت ﻣﺻر اﻟﻘرن اﻟﻌﺷرﯾن ﺗﺣت اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ، اﻟذي ﺗﺳﺑب ﻓﻲ ﺿﻐوط اﻗﺗﺻﺎدﯾﺔ وﺳﯾﺎﺳﯾﺔ ﻛﺑﯾرة. • ﺗﺻﺎﻋدت اﻟﻣﻘﺎوﻣﺔ اﻟﺷﻌﺑﯾﺔ واﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ ﺿد اﻻﺣﺗﻼل، ﺑزﻋﺎﻣﺔ ﻣﺻطﻔﻰ ﻛﺎﻣل وﻣﺣﻣد ﻓرﯾد. • ظﮭرت ﺛورة 1919 ﻟﻠﻣطﺎﻟﺑﺔ ﺑﺎﻻﺳﺗﻘﻼل، وﻛﺎن ﻟﻠزﻋﯾم ﺳﻌد زﻏﻠول دور ﺑﺎرز ﻓﯾﮭﺎ. • ﻓﻲ ﻋﺎم 1922 ، ﺗم إﻋﻼن اﺳﺗﻘﻼل ﻣﺻر، ﻟﻛن اﻟﻧﻔوذ اﻟﺑرﯾطﺎﻧﻲ ظل ﻛﺑﯾرًا. اﻟﻔﺗرة اﻟﺛﺎﻧﯾﺔ ) 1922 - 1952 :( اﻟﺣﻛم اﻟﻣﻠﻛﻲ واﻟﻧﮭﺿﺔ اﻟﺛﻘﺎﻓﯾﺔ • ﺷﮭدت ھذه اﻟﻔﺗرة اﻟﺣﻛم اﻟﻣﻠﻛﻲ، وﺗوﻟﻰ ﻓؤاد اﻷول اﻟﻌرش. • ظﮭور ﺣرﻛﺔ ﺛﻘﺎﻓﯾﺔ واﺳﻌﺔ، ﻣﻊ ظﮭور رواد أدب وﻓن وﺳﯾﺎﺳﺔ. • ﺗﺄﺳﯾس اﻷﺣزاب اﻟﺳﯾﺎﺳﯾﺔ، ﻣﺛل ﺣزب اﻟوﻓد وﺣزب اﻷﻣﺔ. • ﺗﺄﺳﯾس ﺟﻣﺎﻋﺔ اﻹﺧوان اﻟﻣﺳﻠﻣﯾن ﻋﺎم 1928 . • ﺷﮭدت ھذه اﻟﻔﺗرة ﺻﻌودًا ﻓﻲ ﻋدد ﻣن اﻟﻘﺿﺎﯾﺎ، ﻣﺛل إﺻﻼح اﻷوﻗﺎف وﺗﻧﻣﯾﺔ اﻟﺗﻌﻠﯾم. اﻟﻔﺗرة اﻟﺛﺎﻟﺛﺔ ) 1952 - 1961 :( اﻟﺛورة واﻟﻧظﺎم اﻟﺟﻣﮭوري • ﻓﻲ ﻋﺎم 1952 ، ﻗﺎﻣت ﺛورة ﺑﻘﯾﺎدة اﻟﺿﺑﺎط اﻷﺣرار، وﺣل اﻟﻧظﺎم اﻟﻣﻠﻛﻲ. • ﺗﺄﺳﺳت اﻟﺟﻣﮭورﯾﺔ، وﺗوﻟﻰ ﺟﻣﺎل ﻋﺑد اﻟﻧﺎﺻر رﺋﺎﺳﺔ اﻟﺟﻣﮭورﯾﺔ. • ﺗﺄﻣﯾم ﻗﻧﺎة اﻟﺳوﯾس ﻓﻲ ﻋﺎم 1956 ، وﺗﺄﻣﯾم اﻟﺑﻧوك واﻟﺷرﻛﺎت ﻓﻲ ﻋﺎم 1961 . • ﺷﮭدت ھذه اﻟﻔﺗرة ﺗﺣدﯾﺎت ﺳﯾﺎﺳﯾﺔ واﻗﺗﺻﺎدﯾﺔ، ﺑﺎﻹﺿﺎﻓﺔ إﻟﻰ ﺻراﻋﺎت ﺳﯾﺎﺳﯾﺔ ﻣﻊ ﺑﻌض اﻟدول اﻷﺧرى.,
generate exactly 10 multiple-choice questions (one correct answer).
Follow these instructions precisely:

1. Distribute the 10 questions according to these constraints:
   - Difficulty: Easy (3), Average (4), Difficult (3)
   - Bloom Taxonomy: Apply (2), Remember (3), Evaluate (5)
   - Response Time: Short (4), Medium (2), Long (4)
follow the distribution proportions strictly. Use reasonable assumptions if the text lacks enough variety.

2. Each question must include:
   - "question_text": (string)
   - "question_answers": (list of 4 choices)
   - "correct_answer": (one of the above choices)
   - "properties": an object that includes:
       "Difficulty": "Easy" | "Average" | "Difficult",
       "Bloom Taxonomy": "Apply" | "Remember" | "Evaluate",
       "Response Time": "Short" | "Medium" | "Long"

3. Language for all question_text and answers must be Arabic.

4. You have to extract JSON details from pdf according the Pydantic details, containing list of 10 question objects.
Do NOT include any explanation, formatting, or text outside the JSON object.

⚠️ Ensure the distribution rules are strictly followed. Extract content accurately and comply with the format.<end_of_turn>
<start_of_turn>model



<bos><start_of_turn>user
You are a question generation expert.
from this text ﺷﮭد ﺗﺎرﯾﺦ ﻣﺻر ﻓﻲ اﻟﻘرن اﻟﻌﺷرﯾن ﺗﺣوﻻت ﺳﯾﺎﺳﯾﺔ واﺟﺗﻣﺎﻋﯾﺔ ﻛﺑﯾرة، ﺑداﯾﺔ ﻣن اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ وﺻوﻻً إﻟﻰ ﻗﯾﺎم اﻟﺛورة ﻋﺎم 1952 وﺗﺣول ﻣﺻر إﻟﻰ ﺟﻣﮭورﯾﺔ. ﺗﻣﯾز ھذا اﻟﻘرن ﺑظﮭور اﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ اﻟﻣﺻرﯾﺔ، اﻟﻣطﺎﻟﺑﺎت ﺑﺎﻻﺳﺗﻘﻼل، وﺗﺄﺳﯾس أﺣزاب ﺳﯾﺎﺳﯾﺔ، ﺑﺎﻹﺿﺎﻓﺔ إﻟﻰ ﺗطورات اﺟﺗﻣﺎﻋﯾﺔ وﺛﻘﺎﻓﯾﺔ ﻣﮭﻣﺔ. اﻟﻔﺗرة اﻷوﻟﻰ ) 1900 - 1922 :( اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ واﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ • ﺑدأت ﻣﺻر اﻟﻘرن اﻟﻌﺷرﯾن ﺗﺣت اﻻﺣﺗﻼل اﻟﺑرﯾطﺎﻧﻲ، اﻟذي ﺗﺳﺑب ﻓﻲ ﺿﻐوط اﻗﺗﺻﺎدﯾﺔ وﺳﯾﺎﺳﯾﺔ ﻛﺑﯾرة. • ﺗﺻﺎﻋدت اﻟﻣﻘﺎوﻣﺔ اﻟﺷﻌﺑﯾﺔ واﻟﺣرﻛﺔ اﻟوطﻧﯾﺔ ﺿد اﻻﺣﺗﻼل، ﺑزﻋﺎﻣﺔ ﻣﺻطﻔﻰ ﻛﺎﻣل وﻣﺣﻣد ﻓرﯾد. • ظﮭرت ﺛورة 1919 ﻟﻠﻣطﺎﻟﺑﺔ ﺑﺎﻻﺳﺗﻘﻼل، وﻛﺎن ﻟﻠزﻋﯾم ﺳﻌد زﻏﻠول دور ﺑﺎرز ﻓﯾﮭﺎ. • ﻓﻲ ﻋﺎم 1922 ، ﺗم إﻋﻼن اﺳﺗﻘﻼل ﻣﺻر، ﻟﻛن اﻟﻧﻔوذ اﻟﺑرﯾطﺎﻧﻲ ظل ﻛﺑﯾرًا. اﻟﻔﺗرة اﻟﺛﺎﻧﯾﺔ ) 1922 - 1952 :( اﻟﺣﻛم اﻟﻣﻠﻛﻲ واﻟﻧﮭﺿﺔ اﻟﺛﻘﺎﻓﯾﺔ • ﺷﮭدت ھذه اﻟﻔﺗرة اﻟﺣﻛم اﻟﻣﻠﻛﻲ، وﺗوﻟﻰ ﻓؤاد اﻷول اﻟﻌرش. • ظﮭور ﺣرﻛﺔ ﺛﻘﺎﻓﯾﺔ واﺳﻌﺔ، ﻣﻊ ظﮭور رواد أدب وﻓن وﺳﯾﺎﺳﺔ. • ﺗﺄﺳﯾس اﻷﺣزاب اﻟﺳﯾﺎﺳﯾﺔ، ﻣﺛل ﺣزب اﻟوﻓد وﺣزب اﻷﻣﺔ. • ﺗﺄﺳﯾس ﺟﻣﺎﻋﺔ اﻹﺧوان اﻟﻣﺳﻠﻣﯾن ﻋﺎم 1928 . • ﺷﮭدت ھذه اﻟﻔﺗرة ﺻﻌودًا ﻓﻲ ﻋدد ﻣن اﻟﻘﺿﺎﯾﺎ، ﻣﺛل إﺻﻼح اﻷوﻗﺎف وﺗﻧﻣﯾﺔ اﻟﺗﻌﻠﯾم. اﻟﻔﺗرة اﻟﺛﺎﻟﺛﺔ ) 1952 - 1961 :( اﻟﺛورة واﻟﻧظﺎم اﻟﺟﻣﮭوري • ﻓﻲ ﻋﺎم 1952 ، ﻗﺎﻣت ﺛورة ﺑﻘﯾﺎدة اﻟﺿﺑﺎط اﻷﺣرار، وﺣل اﻟﻧظﺎم اﻟﻣﻠﻛﻲ. • ﺗﺄﺳﺳت اﻟﺟﻣﮭورﯾﺔ، وﺗوﻟﻰ ﺟﻣﺎل ﻋﺑد اﻟﻧﺎﺻر رﺋﺎﺳﺔ اﻟﺟﻣﮭورﯾﺔ. • ﺗﺄﻣﯾم ﻗﻧﺎة اﻟﺳوﯾس ﻓﻲ ﻋﺎم 1956 ، وﺗﺄﻣﯾم اﻟﺑﻧوك واﻟﺷرﻛﺎت ﻓﻲ ﻋﺎم 1961 . • ﺷﮭدت ھذه اﻟﻔﺗرة ﺗﺣدﯾﺎت ﺳﯾﺎﺳﯾﺔ واﻗﺗﺻﺎدﯾﺔ، ﺑﺎﻹﺿﺎﻓﺔ إﻟﻰ ﺻراﻋﺎت ﺳﯾﺎﺳﯾﺔ ﻣﻊ ﺑﻌض اﻟدول اﻷﺧرى.,
generate exactly 10 multiple-choice questions (one correct answer).
Follow these instructions precisely:

1. Distribute the 10 questions according to these constraints:
   - Difficulty: Easy (3), Average (4), Difficult (3)
   - Bloom Taxonomy: Apply (2), Remember (3), Evaluate (5)
   - Response Time: Short (4), Medium (2), Long (4)
follow the distribution proportions strictly. Use reasonable assumptions if the text lacks enough variety.

2. Each question must include:
   - "question_text": (string)
   - "question_answers": (list of 4 choices)
   - "correct_answer": (one of the above choices)
   - "properties": an object that includes:
       "Difficulty": "Easy" | "Average" | "Difficult",
       "Bloom Taxonomy": "Apply" | "Remember" | "Evaluate",
       "Response Time": "Short" | "Medium" | "Long"

3. Language for all question_text and answers must be Arabic.

4. You have to extract JSON details from pdf, containing list of 10 question objects.
Do NOT include any explanation, formatting, or text outside the JSON object.

⚠️ Ensure the distribution rules are strictly followed. Extract content accurately and comply with the format.<end_of_turn>
<start_of_turn>model


In [ ]:
import torch
dtype = torch.bfloat16
quantization_config = BitsAndBytesConfig(
          load_in_4bit=True,
          bnb_4bit_compute_dtype=torch.float16
      )
model_name = 'google/gemma-7b-it'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=quantization_config
                                             ,torch_dtype=dtype,
                                             device_map="auto",)

ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [ ]:
!pip uninstall -y bitsandbytes
!pip install bitsandbytes==0.46.1

Found existing installation: bitsandbytes 0.46.1
Uninstalling bitsandbytes-0.46.1:
  Successfully uninstalled bitsandbytes-0.46.1
  Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl (72.9 MB)


In [ ]:
outputs = model.generate(input_ids=inputs["input_ids"].to(model.device),
                              max_new_tokens=2048)
generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]